# 🌱 Rootzone Predictor — pH & EC

### How to use
1. **Cell 1** — Set the path to your CSV file (and optionally a target time)
2. **Run All Cells** (`Kernel → Restart & Run All`)
3. Read the prediction at the bottom

---

### CSV format
| timestamp | ph | ec_ms | irrigation_ml_current | ET0 | ... |
|---|---|---|---|---|---|
| 2025-09-30 08:00 | **9.2** | **0.35** | 500 | 0.12 | ... |
| 2025-09-30 10:00 | | | 200 | 0.10 | ... |
| 2025-10-01 08:00 | | | 400 | 0.15 | ... |

- **First row with ph + ec filled in** → used as the anchor (last known measurement)
- **All other rows** → sensor readings between anchor and prediction time
- **Target time** → where you want the prediction (default: last row in the file)

In [ ]:
# ============================================================
#  ✏️  EDIT ONLY THIS CELL
# ============================================================

# Path to your CSV file
CSV_FILE = 'greenhouse_data.csv'

# Target time for the prediction.
# Set to None to predict at the last row of the CSV.
TARGET_TIME = None
# TARGET_TIME = '2025-10-01 14:00'

# Path to the saved model folder (created by the save cell in the training notebook)
MODEL_DIR = 'saved_model'
# ============================================================

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import json, sys
import numpy as np
import pandas as pd
import joblib

# ── Fertilizer column names (must match your CSV headers) ────
ACID_FERTS      = ['Phosphoric acid[mg]-H3PO4']
SALT_FERTS      = ['Monopotassium Phosphate[mg] -KH2PO4', 'Potassium Chloride[mg] - KCL',
                   'Kortin [mg]', 'Ammonium Nitrate [mg] -NH4NO3', 'Gypsum - CaSO4*2H2O [mg]']
CORE_SALT_FERTS = ['Monopotassium Phosphate[mg] -KH2PO4', 'Potassium Chloride[mg] - KCL',
                   'Ammonium Nitrate [mg] -NH4NO3']

print('✅ Libraries loaded')

In [ ]:
# ── Internal helpers (no need to edit) ───────────────────────
def _to_num(s, d=0.0):      return pd.to_numeric(s, errors='coerce').fillna(d)
def _get_fert_any(seg):     return _sum_series(seg, ACID_FERTS + SALT_FERTS)

def _sum(fr, cols):
    u = [c for c in cols if c in fr.columns]
    return float(fr[u].apply(pd.to_numeric, errors='coerce').fillna(0).sum().sum()) if u else 0.0

def _sum_series(fr, cols):
    u = [c for c in cols if c in fr.columns]
    return fr[u].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1) if u else pd.Series(0.0, index=fr.index)

def _seg(df, start, stop):
    if pd.Timestamp(start) >= pd.Timestamp(stop): return df.iloc[0:0]
    return df.loc[(df.index >= pd.Timestamp(start)) & (df.index < pd.Timestamp(stop))]


def extract_features(df, anchor_ts, target_ts):
    t0  = pd.Timestamp(anchor_ts)
    t1  = pd.Timestamp(target_ts)
    seg = _seg(df, t0, t1)

    ph0   = float(df.loc[t0, 'ph'])
    ec0   = float(df.loc[t0, 'ec_ms'])
    gap_h = float((t1 - t0).total_seconds() / 3600.0)

    irr_s           = _to_num(seg['irrigation_ml_current']) if 'irrigation_ml_current' in seg.columns else pd.Series(0.0, index=seg.index)
    irr_total       = float(irr_s.sum())
    fert_salt_total = _sum(seg, SALT_FERTS)
    h3po4_total     = _sum(seg, ACID_FERTS)
    ET0_sum         = float(_to_num(seg['ET0']).sum()) if 'ET0' in seg.columns else 0.0

    if 'internal_air_temp_c' in seg.columns and 'internal_rh_%' in seg.columns and len(seg) > 0:
        temp_s   = _to_num(seg['internal_air_temp_c'])
        rh_s     = _to_num(seg['internal_rh_%'])
        es_s     = 0.6108 * np.exp((17.27 * temp_s) / (temp_s + 237.3))
        vpd_mean = float((es_s * (1 - rh_s / 100)).mean())
    else:
        temp_s   = pd.Series(dtype=float)
        vpd_mean = 0.0

    soil_temp_mean = float(_to_num(seg['soil_temp_pred']).mean())  if 'soil_temp_pred'      in seg.columns and len(seg) > 0 else 0.0
    canopy         = float(_to_num(seg['canopy_cover']).mean())     if 'canopy_cover'         in seg.columns and len(seg) > 0 else 0.0
    rad_morning    = float(_to_num(seg.loc[(seg.index.hour >= 6) & (seg.index.hour < 12), 'internal_radiation']).sum()) \
                     if 'internal_radiation' in seg.columns and len(seg) > 0 else 0.0

    ft             = _get_fert_any(seg); ft = ft[ft > 0].index
    hrs_since_fert = float((t1 - ft.max()).total_seconds() / 3600) if len(ft) > 0 else gap_h

    cs             = _sum_series(seg, CORE_SALT_FERTS)
    sei            = cs[cs > 0].index.max() if len(cs[cs > 0]) else None
    last_salt_dose = float(cs.loc[sei])                                            if sei is not None else 0.0
    last_irr_amt   = float(irr_s.loc[sei]) if sei is not None and sei in irr_s.index else 0.0
    hrs_since_salt = float((t1 - sei).total_seconds() / 3600)                     if sei is not None else gap_h
    last_ev_conc   = float(last_salt_dose / (last_irr_amt + 1.0))
    irr_after_salt = float(_to_num(seg.loc[seg.index > sei, 'irrigation_ml_current']).sum()) \
                     if sei is not None and 'irrigation_ml_current' in seg.columns else 0.0

    r6h  = _seg(df, t1 - pd.Timedelta(hours=6),  t1)
    p24h = _seg(df, t1 - pd.Timedelta(hours=24), t1 - pd.Timedelta(hours=12))
    h24  = _seg(df, t1 - pd.Timedelta(hours=24), t1)
    pd48 = _seg(df, t1 - pd.Timedelta(hours=48), t1 - pd.Timedelta(hours=24))
    pr48 = _seg(df, t0 - pd.Timedelta(hours=48), t0)

    irr_r   = float(_to_num(r6h['irrigation_ml_current']).sum())    if 'irrigation_ml_current' in r6h.columns  and len(r6h)  > 0 else 0.0
    irr_p   = float(_to_num(p24h['irrigation_ml_current']).sum())   if 'irrigation_ml_current' in p24h.columns and len(p24h) > 0 else 0.0
    irr_pd  = float(_to_num(pd48['irrigation_ml_current']).sum())   if 'irrigation_ml_current' in pd48.columns and len(pd48) > 0 else 0.0
    dark_6h = float((_to_num(r6h['internal_radiation']) <= 10).sum()) / 6.0 \
              if 'internal_radiation' in r6h.columns and len(r6h) > 0 else 0.0

    salt_buildup    = 0.0
    hrs_since_fert2 = 24.0
    if len(h24) > 1:
        hi_irr  = _to_num(h24['irrigation_ml_current']) if 'irrigation_ml_current' in h24.columns else pd.Series(0.0, index=h24.index)
        ss      = _sum_series(h24, SALT_FERTS); sm = ss > 0
        if sm.any():
            hb          = np.array((t1 - h24.index[sm]).total_seconds()) / 3600
            salt_buildup = float((ss[sm].values * np.exp(-0.15 * hb)).sum()) - float(hi_irr.sum()) * 0.1
        fa2 = _get_fert_any(h24); ft2 = h24.index[fa2 > 0]
        hrs_since_fert2 = float((t1 - ft2[-1]).total_seconds() / 3600) if len(ft2) > 0 else 24.0

    salt48 = float(_sum(pr48, SALT_FERTS))

    prev_ec_slope = 0.0
    before = df.loc[df.index < t0]
    lab    = before[before['ph'].notna() & before['ec_ms'].notna()]
    if len(lab) >= 1:
        t_prev        = lab.index[-1]
        prev_gap_h    = max((t0 - t_prev).total_seconds() / 3600, 1e-6)
        prev_ec_slope = float((ec0 - float(df.loc[t_prev, 'ec_ms'])) / prev_gap_h)

    t1_soil = float(_to_num(df.loc[t1:t1, 'soil_temp_pred']).iloc[0])  if t1 in df.index and 'soil_temp_pred'      in df.columns else 0.0
    t0_soil = float(_to_num(df.loc[t0:t0, 'soil_temp_pred']).iloc[0])  if               'soil_temp_pred'      in df.columns else 0.0
    rad_t1  = float(_to_num(df.loc[t1:t1, 'internal_radiation']).iloc[0]) if t1 in df.index and 'internal_radiation' in df.columns else 0.0
    days_ap = float(df.loc[t1, 'days_after_planting']) if 'days_after_planting' in df.columns and t1 in df.index else 0.0

    salt_conc = float(fert_salt_total / (irr_total + 1.0))
    hour_b    = t1.hour + t1.minute / 60.0
    safe_h    = max(gap_h, 0.25)

    return {
        'ph0': ph0, 'ec0': ec0, 'gap_hours': gap_h,
        'irr_total_t0_t1': irr_total, 'fert_salt_total_t0_t1': fert_salt_total,
        'ET0_sum_t0_t1': ET0_sum, 'ET0_per_hour': float(ET0_sum / max(gap_h, 0.16)),
        'h3po4_total': h3po4_total,
        'log_ec_drive': float(np.log((fert_salt_total + h3po4_total) / (irr_total + 1) + 0.01) - np.log(ec0 + 0.01)),
        'soil_temp_mean': soil_temp_mean, 'transpiration_pull': float(vpd_mean * canopy),
        'photo_temp_interaction': float(vpd_mean * soil_temp_mean),
        'temp_x_canopy': float(soil_temp_mean * canopy), 'salt_conc_t0_t1': salt_conc,
        'irr_to_et0': float(irr_total / (ET0_sum + 0.1)),
        'stage_x_salt_conc': float((days_ap / 100.0) * salt_conc),
        'ec_log_anchor': float(np.log(ec0 + 0.05)), 'hrs_since_fert': hrs_since_fert,
        'hrs_since_last_salt_event': hrs_since_salt, 'last_event_salt_conc': last_ev_conc,
        'irr_after_last_salt': irr_after_salt,
        'temp_trend': float(temp_s.iloc[-1] - temp_s.iloc[0]) if len(temp_s) > 1 else 0.0,
        'hour_sin_b': float(np.sin(2 * np.pi * hour_b / 24.0)),
        'hour_cos_b': float(np.cos(2 * np.pi * hour_b / 24.0)),
        't1_morning_short': int((5 <= t1.hour <= 10) and (gap_h <= 1.5)),
        't1_early_day': int(5 <= hour_b < 13.0),
        'soil_delta': float(t1_soil - t0_soil), 'rad_t1_log': float(np.log1p(rad_t1)),
        'rad_morning': rad_morning, 'hist_irr_recent': irr_r, 'hist_irr_prior': irr_p,
        'hist_dark_recent_6h': dark_6h, 'hist_salt_buildup': salt_buildup,
        'hist_hrs_since_fert': hrs_since_fert2, 'hist48_irr_prevday': irr_pd,
        'prev_ec_slope': prev_ec_slope,
        'salt_recency_pressure': float(last_ev_conc * np.exp(-hrs_since_salt / 6.0)),
        'salt_event_pos': float(np.clip(1.0 - (hrs_since_salt / safe_h), 0.0, 1.0)),
        'ec0_x_p48_salt': float(ec0 * salt48),
    }


def run_model(feats, model, meta):
    EC_SHIFT = meta['EC_TARGET_SHIFT']
    blend    = np.array([meta['ROBUST_LINEAR_BLEND_PH'], meta['ROBUST_LINEAR_BLEND_EC']])
    gate_h   = meta['ROBUST_LINEAR_GATE_MAX_GAP_H']
    X        = pd.DataFrame([feats])[meta['feature_cols']].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    base = np.asarray(model['base'].predict(X))[0] * model['target_std'] + model['target_mean']
    rl   = model.get('robust_linear')
    gate = bool(
        feats.get('fert_salt_total_t0_t1', 0) >= 250 and feats.get('irr_after_last_salt', 0) <= 1 and
        feats.get('salt_conc_t0_t1',       0) >= 3   and feats.get('ec0', np.inf)              <= 0.8 and
        feats.get('gap_hours', np.inf) <= gate_h
    )
    if rl is not None and gate:
        rl_raw = np.asarray(rl.predict(X))[0] * model['target_std'] + model['target_mean']
        final  = (1.0 - blend) * base + blend * rl_raw
    else:
        final  = base

    ph_pred = feats['ph0'] + final[0]
    ec_pred = max(0.0, (feats['ec0'] + EC_SHIFT) * np.exp(final[1]) - EC_SHIFT)
    return round(float(ph_pred), 3), round(float(ec_pred), 4)

print('✅ Feature extraction and model functions ready')

In [ ]:
# ── Load model ────────────────────────────────────────────────
try:
    model = joblib.load(f'{MODEL_DIR}/v8_shared_model.joblib')
    with open(f'{MODEL_DIR}/v8_model_meta.json') as f:
        meta = json.load(f)
    print(f'✅ Model loaded  (trained on {meta["n_train"]} rows, saved {meta["saved_at"][:10]})')
except FileNotFoundError:
    print(f'❌ Model not found in "{MODEL_DIR}/"')
    print('   Run the save cell in your training notebook first.')
    raise

# ── Load CSV ──────────────────────────────────────────────────
try:
    df = pd.read_csv(CSV_FILE, parse_dates=['timestamp']).sort_values('timestamp').set_index('timestamp')
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    print(f'✅ CSV loaded  ({len(df)} rows, {df.index[0].date()} → {df.index[-1].date()})')
except FileNotFoundError:
    print(f'❌ CSV file not found: "{CSV_FILE}"')
    print('   Check that CSV_FILE points to the right path.')
    raise

# ── Find anchor ───────────────────────────────────────────────
labeled = df[df['ph'].notna() & df['ec_ms'].notna()]
if labeled.empty:
    raise ValueError('❌ No row with both ph and ec_ms filled in. The anchor row is missing.')

anchor_ts = labeled.index[0]   # first labeled row = anchor
print(f'✅ Anchor found  → {anchor_ts}  (pH={df.loc[anchor_ts, "ph"]:.3f}, EC={df.loc[anchor_ts, "ec_ms"]:.4f} mS/cm)')

# ── Set target time ───────────────────────────────────────────
if TARGET_TIME is not None:
    target_ts = pd.Timestamp(TARGET_TIME)
    if target_ts not in df.index:
        placeholder = pd.DataFrame(index=[target_ts], columns=df.columns, dtype=float)
        df = pd.concat([df, placeholder]).sort_index()
else:
    target_ts = df.index[-1]

print(f'✅ Target time   → {target_ts}')

In [ ]:
# ── Run prediction ────────────────────────────────────────────
feats            = extract_features(df, anchor_ts, target_ts)
ph_pred, ec_pred = run_model(feats, model, meta)

print()
print('=' * 50)
print('  ROOTZONE PREDICTION')
print('=' * 50)
print(f'  Anchor  : {anchor_ts.strftime("%Y-%m-%d %H:%M")}')
print(f'            pH = {feats["ph0"]:.3f}   EC = {feats["ec0"]:.4f} mS/cm')
print(f'  Target  : {target_ts.strftime("%Y-%m-%d %H:%M")}')
print(f'  Gap     : {feats["gap_hours"]:.1f} hours')
print('-' * 50)
print(f'  Predicted pH  :  {ph_pred}')
print(f'  Predicted EC  :  {ec_pred} mS/cm')
print('=' * 50)